# 2PL_DQN — Google Colab 実行ノートブック

リポジトリ: <https://github.com/ituki0426/2PL_DQN>

このノートブックは `run_grid.py` / `benchmark_cost.py` / `analyze_selection.py` / `make_tables.py` を Colab から順に実行するためのものです。

**推奨設定**: `ランタイム → ランタイムのタイプを変更 → GPU`。EXP004 は EXP001 を基にした CUDA・FP64 版（能力推定は MLE＋Dodd）です。CUDA が使えない場合は自動で CPU に切り替えます。
DQN の学習・推論が GPU の対象で、能力推定・反応生成は CPU で実行します。

**実験**: EXP001–EXP004（シミュレーション）はセクション 3–7 の共通フロー、**EXP005（実データ・実受験者応答）はセクション 8** の専用フロー（`--dataset` 指定、rep ごとに分割再サンプル）で実行します。

**セッション制限に注意**: 無料枠は最大 12h、90 分程度アイドルで切断。replication は独立プロセスなので、切れたら該当 rep だけ再実行すれば続きから進められます。

## 1. セットアップ — リポジトリ取得と依存関係のインストール

In [2]:
import os
!git clone https://github.com/ituki0426/2PL_DQN.git
%cd 2PL_DQN
!pip install -q -r requirements.txt

Cloning into '2PL_DQN'...
remote: Enumerating objects: 183, done.
remote: Counting objects: 100% (183/183), done.
remote: Compressing objects: 100% (129/129), done.
remote: Total 183 (delta 63), reused 166 (delta 46), pack-reused 0 (from 0)
Receiving objects: 100% (183/183), 700.33 KiB | 26.94 MiB/s, done.
Resolving deltas: 100% (63/63), done.
/content/2PL_DQN/2PL_DQN


In [3]:
import torch, numpy, pandas, matplotlib
print('torch', torch.__version__, 'CUDA build', torch.version.cuda)
if torch.cuda.is_available():
    print('EXP004 DQN device: cuda /', torch.cuda.get_device_name(0))
else:
    print('EXP004 DQN device: cpu（CUDA が利用できないため自動切り替え）')
print('EXP004 DQN dtype:', torch.float64)
print('numpy', numpy.__version__)
print('pandas', pandas.__version__)
print('matplotlib', matplotlib.__version__)


torch 2.11.0+cu128 CUDA build 12.8
EXP004 DQN device: cuda / NVIDIA L4
EXP004 DQN dtype: torch.float64
numpy 2.1.3
pandas 2.2.3
matplotlib 3.10.0


## 2. （任意）Google Drive をマウント

セッションが切れても結果を残したい場合に使います。使わない場合はこのセルをスキップ。

In [4]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_OUT = '/content/drive/MyDrive/2PL_DQN_results'
!mkdir -p "$DRIVE_OUT"
print('output dir:', DRIVE_OUT)

Mounted at /content/drive
output dir: /content/drive/MyDrive/2PL_DQN_results


## 3. スモークテスト（`--quick`）

`--quick` は `n_episodes=200` に短縮して数分で完走します。まずここで環境確認。

In [ ]:
# 実験を選択（セクション 3–7 で使用。EXP005 はセクション 8 の専用フローを使う）
EXPERIMENT = "EXP004"  # "EXP001" | "EXP002" | "EXP003" | "EXP004"
!python run_grid.py --list-experiments

!python run_grid.py --experiment $EXPERIMENT --grid main --rep 4 --quick --out /tmp/2PL_DQN_smoke

### （任意）EXP004 のデバイス動作確認

CPU への自動切り替えと DQN の FP64 学習（バッファ・損失・勾配を含む）を確認します。CUDA が利用できる場合は、GPU 上の学習・推論・回答済み項目の除外もテストします。CUDA がない場合、GPU のテストはスキップされます。

In [6]:
!python -m unittest discover -s tests -p test_exp004_cuda.py -v

test_cpu_fallback_fp64_training_and_inference (test_exp004_cuda.DQNDeviceTests.test_cpu_fallback_fp64_training_and_inference) ... ok
test_cuda_fp64_training_and_inference (test_exp004_cuda.DQNDeviceTests.test_cuda_fp64_training_and_inference) ... ok
test_features_preserve_fp64_precision (test_exp004_cuda.DQNDeviceTests.test_features_preserve_fp64_precision) ... ok

----------------------------------------------------------------------
Ran 3 tests in 1.953s

OK


## 4. `run_grid.py` — 本番グリッド実行

各グリッドで走らせる replication 番号:

| グリッド | reps | 概要 |
| --- | --- | --- |
| `main` | 4, 5, 6 | 既存 vs 提案（表 tab:main） |
| `guess` | 4, 5, 6 | 3PLM 応答（表 tab:guess） |
| `sensitivity` | 1, 2, 3 | 報酬 × γ の網羅（表 tab:sensitivity） |
| `state` | 1, 2, 3 | 状態 A/B/C（表 tab:state） |
| `ablation` | 1, 2, 3 | 一因子入れ替え（表 tab:ablation） |

1 条件あたりの学習は重いです。無料 GPU で完走が難しいときは `--conditions proposed` などで条件を絞ってください。

In [7]:
# ここで対象グリッドと rep を選ぶ
GRID = 'main'          # 'main' | 'sensitivity' | 'state' | 'ablation' | 'guess'
REPS = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]        # main / guess は 4-6、それ以外は 1-3
EXTRA_ARGS = ''         # 例: '--conditions proposed' や '--quick'

In [ ]:
for rep in REPS:
    print(f'\n=== {GRID}: rep {rep} ===', flush=True)
    !python run_grid.py --experiment $EXPERIMENT --grid $GRID --rep $rep $EXTRA_ARGS


=== main: rep 1 ===
rep 1: asymptotic reference value from true-theta information 0.216
rep 1: MFI                    RMSE@40 0.218  items 203
rep 1: FIWL                   RMSE@40 0.219  items 195
rep 1: MPWI                   RMSE@40 0.220  items 183
rep 1: MEPV                   RMSE@40 0.217  items 184
rep 1: existing DQN device: cuda, dtype: torch.float64
rep 1: existing               RMSE@40 0.262  items 73  val return 16.475  updates 39873  176s
rep 1: proposed DQN device: cuda, dtype: torch.float64
rep 1: proposed               RMSE@40 0.218  items 166  val return 21.170  updates 24997  199s

=== main: rep 2 ===
rep 2: asymptotic reference value from true-theta information 0.215
rep 2: MFI                    RMSE@40 0.221  items 194
rep 2: FIWL                   RMSE@40 0.221  items 188
rep 2: MPWI                   RMSE@40 0.215  items 183
rep 2: MEPV                   RMSE@40 0.218  items 184
rep 2: existing DQN device: cuda, dtype: torch.float64
rep 2: existing             

In [ ]:
# 全 rep が揃ったら mean.csv と mean.png（横軸 step）を保存
!python run_grid.py --experiment $EXPERIMENT --grid $GRID --aggregate

from IPython.display import Image, display
display(Image(filename=f'result/{EXPERIMENT}/{GRID}/mean.png'))


## 5. `benchmark_cost.py` — 選択規則の推論コスト計測

`result/<実験名>/cost/cost.csv` を出力します。CPU は単一スレッド固定です。EXP004 の DQN は CUDA が利用できれば GPU で計測します（CPU とのデータ転送を含む）。

In [ ]:
!python benchmark_cost.py --experiment $EXPERIMENT

## 6. `analyze_selection.py` — 選択アイテムの分析＋図出力

`result/<実験名>/selection/` に CSV と `fig_selection.pdf` を出力します。

In [ ]:
!python analyze_selection.py --experiment $EXPERIMENT

In [ ]:
# PDF をノートブック内で確認したい場合
from IPython.display import IFrame
IFrame(f'result/{EXPERIMENT}/selection/fig_selection.pdf', width=800, height=500)

## 7. `make_tables.py` — LaTeX 表行の生成

対応するグリッドの `mean.csv` / `rep*.csv` が揃っている必要があります。EXP001 の計算済み結果は `result/EXP001/` に含まれています。EXP002・EXP003・EXP004 は実行後に利用できます。

In [ ]:
for name in ['main', 'sensitivity', 'state', 'ablation', 'guess', 'cost']:
    print(f'\n===== {name} =====')
    !python make_tables.py --experiment $EXPERIMENT $name

## 8. EXP005（実データ・実受験者応答）実行

EXP005 は Wang, Liu, & Xu (2024) のシミュレーションではなく、**実受験者データ**に対して DQN 手法を評価する実験です。EXP001–004 と CLI が異なる点：

- `--dataset <name>` が必須（`--grid` は使わない）
- `--rep 1..5`。分割シードは `--split-seed + rep - 1` で **rep ごとに train/validation/test を再サンプル**（`existing/proposed` に加え `MFI/FIWL/MPWI/MEPV` も rep ごとに評価されるので、`rep{N}.csv` に 6 条件すべてが入る）
- DQNの `existing` / `proposed` はともに `n_env=32` で実回答を並列処理します
- 集計後に、選択中のデータセットを使って `benchmark_cost.py` / `analyze_selection.py` / `make_tables.py` も実行します。選択分析には `EXP005_REPS` の先頭の proposed モデルを使います

対応データセット: `choi_2026_cmsce_2019_2`, `choi_2026_cmsce_2020_1`, `choi_2026_cmsce_2021_2`

### データの準備

リポジトリには実データは含まれていません（`data/` は git 未追跡）。次のいずれかで用意してください。

**A. Google Drive 経由（推奨）**: `MyDrive/2PL_DQN_data/<dataset>/` に次の 3 ファイルを配置：

| ファイル | 必須カラム |
|---|---|
| `item_parameters.csv` | `item, a, b` |
| `person_scores.csv` | `id, theta_EAP` |
| `responses.csv` | `id` と各項目IDカラム（値は 0/1） |

セクション 2 で Drive をマウント済みなら、次のセルで `data/` にコピーします。

**B. 直接アップロード**: `from google.colab import files; files.upload()` で個別に上げて `data/<dataset>/` に配置。

In [ ]:
# Google Drive → リポジトリ内 data/ にコピー（セクション 2 で Drive をマウント済み前提）
import os, shutil
DRIVE_DATA = '/content/drive/MyDrive/2PL_DQN_data'
EXP005_DATASETS = ('choi_2026_cmsce_2019_2', 'choi_2026_cmsce_2020_1', 'choi_2026_cmsce_2021_2')
os.makedirs('data', exist_ok=True)
for ds in EXP005_DATASETS:
    src, dst = f'{DRIVE_DATA}/{ds}', f'data/{ds}'
    if not os.path.isdir(src):
        print(f'✗ {ds}: {src} が見つかりません — スキップ')
        continue
    os.makedirs(dst, exist_ok=True)
    for name in ('item_parameters.csv', 'person_scores.csv', 'responses.csv'):
        s = f'{src}/{name}'
        if os.path.exists(s):
            shutil.copy(s, dst)
    print(f'✓ {ds}: {sorted(os.listdir(dst))}')

In [ ]:
# 実行するデータセットと rep を選択
EXP005_DATASET = 'choi_2026_cmsce_2019_2'   # 'choi_2026_cmsce_2019_2' | 'choi_2026_cmsce_2020_1' | 'choi_2026_cmsce_2021_2'
EXP005_REPS = [1]                            # 例: [1, 2, 3, 4, 5] で全 rep
EXP005_EXTRA = ''  # 例: '--quick'（128/64/64・1 epoch）、'--conditions proposed'、'--out /content/drive/MyDrive/2PL_DQN_results'

In [ ]:
for rep in EXP005_REPS:
    print(f'\n=== EXP005 / {EXP005_DATASET} / rep {rep} ===', flush=True)
    !python run_grid.py --experiment EXP005 --dataset $EXP005_DATASET --rep $rep $EXP005_EXTRA

In [ ]:
# 全 rep 完了後の集計。mean.csv と mean.png を出力する
!python run_grid.py --experiment EXP005 --dataset $EXP005_DATASET --aggregate

from IPython.display import Image, display
display(Image(filename=f'result/EXP005/{EXP005_DATASET}/main/mean.png'))

### EXP005 の推論コスト・選択項目分析・表生成

`benchmark_cost.py` は実データの項目バンクとテスト回答を使って推論コストを計測します。`analyze_selection.py` は実行済みの proposed モデルを読み込み、MFI・MEPV と選択項目を比較します。最後に `make_tables.py` で `main` の LaTeX 表行を生成します。

In [ ]:
EXP005_ANALYSIS_REP = EXP005_REPS[0]

!python benchmark_cost.py --experiment EXP005 --dataset $EXP005_DATASET --rep $EXP005_ANALYSIS_REP
!python analyze_selection.py --experiment EXP005 --dataset $EXP005_DATASET --rep $EXP005_ANALYSIS_REP
!python make_tables.py --experiment EXP005 --dataset $EXP005_DATASET main

from IPython.display import IFrame
IFrame(f'result/EXP005/{EXP005_DATASET}/selection/fig_selection.pdf', width=800, height=500)

## 9. 結果を Google Drive に保存（マウント済みの場合）

`result/` 以下は EXP001–EXP005 すべての出力を含みます（EXP005 は `result/EXP005/<dataset>/` 配下の `main/`・`cost/`・`selection/` に保存されます）。

In [ ]:
!cp -r result "$DRIVE_OUT/"
!ls "$DRIVE_OUT/result"

In [ ]:
# 選択した実験の保存先を確認
!ls result/$EXPERIMENT